In [93]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
from shapely import wkt
import warnings
# filter all warnings
warnings.filterwarnings("ignore")

aquestes parades no tenen les linies. 
el json de https://ide.amb.cat/geoserveis/rest/services/ADSE/InfoPAE_AUX/MapServer no té les posicions

https://www.trenscat.com/tram/tramt1_ct.html

In [94]:
pd.read_excel("Data/Tram/tram_stops.xlsx")


,Name,Latitude,Longitude,OutboundCode,ReturnCode,GtfsCode,Connections (Id)
0,Francesc Macià,41.392200,2.143175,1001,1101,FRMC,NaN
1,L'Illa,41.390560,2.136497,1002,1102,ILLA,NaN
2,Numància,41.389339,2.131953,1003,1103,NMNC,NaN
3,Maria Cristina,41.387970,2.126508,1004,1104,MRCR,15
4,Pius XII,41.386719,2.121603,1005,1105,PIUS,NaN
5,Palau Reial,41.385921,2.118461,1008,1108,PLRL,16
6,Zona Universitària,41.384270,2.114436,1009,1109,UNIV,21
7,Avinguda de Xile,41.380741,2.114836,1010,1110,XILE,NaN
8,Ernest Lluch,41.376480,2.110786,1011,1111,LLUC,17
9,Can Rigal,41.376122,2.105928,1012,1112,SRMN,NaN


In [95]:
stops_w_points = pd.read_excel("Data/Tram/tram_stops.xlsx")
stops_w_points = stops_w_points[['Name', 'Latitude', 'Longitude','GtfsCode']]
geo_df_tram_stops = gpd.GeoDataFrame(stops_w_points, geometry=gpd.points_from_xy(stops_w_points.Longitude, stops_w_points.Latitude))
geo_df_tram_stops.drop(columns=['Latitude', 'Longitude'], inplace=True)
geo_df_tram_stops['stop_type'] = 'Tram'
geo_df_tram_stops.rename(columns={'Name': 'name', 'GtfsCode': 'id'}, inplace=True)


# Solutions

In [96]:
solutions = geo_df_tram_stops.copy()
already_in_metro = ['Ernest Lluch','Zona Universitària','Palau Reial','Maria Cristina','Fòrum','El Maresme','Selva de Mar','Glòries','Marina','Ciutadella | Vila Olímpica','Gorg','Sant Roc','Besòs','Verdaguer']
solutions = solutions[~solutions['name'].isin(already_in_metro)]
solutions['id'] = 'ST' + '-' + solutions['id'].astype(str)
solutions = solutions[['id', 'name', 'stop_type','geometry']]

In [97]:
solutions.to_csv('Nodes/Tram-Sol.csv', index=False)

# Tram Nodes

In [98]:
stops_map = {
    'Bon Viatge': ['T1', 'T2'],
    'Fontsanta | Fatjó': ['T1', 'T2'],
    'Les Aigües': ['T1', 'T2'],
    'Cornellà Centre': ['T1', 'T2'],
    'Ignasi Iglésias': ['T1', 'T2'],
    'El Pedró': ['T1', 'T2'],
    'Montesa': ['T1', 'T2', 'T3'],
    'La Sardana': ['T1', 'T2', 'T3'],
    "Pont d'Esplugues": ['T1', 'T2', 'T3'],
    'Can Clota': ['T1', 'T2', 'T3'],
    "Ca n'Oliveres": ['T1', 'T2', 'T3'],
    'Can Rigal': ['T1', 'T2', 'T3'],
    'Ernest Lluch': ['T1', 'T2', 'T3'],
    'Avinguda de Xile': ['T1', 'T2', 'T3'],
    'Zona Universitària': ['T1', 'T2', 'T3'],
    'Palau Reial': ['T1', 'T2', 'T3'],
    'Pius XII': ['T1', 'T2', 'T3'],
    'Maria Cristina': ['T1', 'T2', 'T3'],
    'Numància': ['T1', 'T2', 'T3'],
    "L'Illa": ['T1', 'T2', 'T3'],
    'Francesc Macià': ['T1', 'T2', 'T3'],
    'La Fontsanta': ['T2'],
    'Centre Miquel Martí i Pol': ['T2'],
    'Llevant - Les planes': ['T2'],
    'Hospital Sant Joan Despí | TV3': ['T3'],
    'Rambla de Sant Just': ['T3'],
    'Walden': ['T3'],
    'Torreblanca': ['T3'],
    'Sant Feliu | Consell Comarcal': ['T3'],
    'Verdaguer': ['T4'],
    'Sicília': ['T4'],
    'Monumental': ['T4'],
    'Glòries': ['T4', 'T5', 'T6'],
    "Ca l'Aranyó": ['T4'],
    'Pere IV': ['T4'],
    'Fluvià': ['T4'],
    'Selva de Mar': ['T4'],
    'El Maresme': ['T4'],
    'Fòrum': ['T4'],
    'Campus Diagonal - Besòs': ['T4'],
    'Port Fòrum': ['T4', 'T6'],
    'Estació de Sant Adrià': ['T4', 'T6'],
    'Ciutadella | Vila Olímpica': ['T5', 'T6'],
    'Wellington': ['T5', 'T6'],
    'Marina': ['T5', 'T6'],
    'Auditori | Teatre Nacional': ['T5', 'T6'],
    'Can Jaumandreu': ['T5', 'T6'],
    'Espronceda': ['T5', 'T6'],
    'Sant Martí de Provençals': ['T5', 'T6'],
    'Besòs': ['T5', 'T6'],
    'Alfons el Magnànim': ['T5', 'T6'],
    'Parc del Besòs': ['T5', 'T6'],
    'La Catalana': ['T5'],
    'Sant Joan Baptista': ['T5'],
    'Encants de Sant Adrià': ['T5'],
    'Sant Roc': ['T5'],
    'Gorg': ['T5'],
    'La Mina': ['T6']
}
keys = set(stops_map.keys())

fer servir el mateix punt per les parades que comparteixen metro i tram

In [99]:
geo_df_tram_stops['linia'] = geo_df_tram_stops['name'].map(stops_map)
geo_df_tram_stops = geo_df_tram_stops.explode('linia', ignore_index=True)
geo_df_tram_stops= geo_df_tram_stops[['id','name','linia','stop_type','geometry']]
geo_df_tram_stops['id'] = 'T' + '-' + geo_df_tram_stops['linia'] + '-' + geo_df_tram_stops['id']


In [100]:
geo_df_tram_stops.to_csv('Nodes/Tram.csv', index=False)